###Setup and imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = 'NLP/Thesis'

import os
os.chdir(f'/content/drive/MyDrive/{path}')
print(f"Working directory: {os.getcwd()}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/NLP/Thesis


In [ ]:
from huggingface_hub import login as hf_login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
hf_login(token=hf_token)

In [ ]:
import json
import math
import re
import gc
import time
import random
import pickle
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Union, Tuple
from abc import ABC, abstractmethod
from io import BytesIO
from difflib import SequenceMatcher

import requests
from PIL import Image
from tqdm import tqdm
import numpy as np
import torch

from transformers import (
    Gemma3ForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    AutoModel,
    AutoTokenizer,
    pipeline,
)

In [ ]:
# Install from source (as recommended on the page)
%pip install git+https://github.com/huggingface/transformers accelerate
%pip install qwen-vl-utils[decord]==0.0.8

# Qwen utils
from qwen_vl_utils import process_vision_info

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-wvd8rp59
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-wvd8rp59
  Resolved https://github.com/huggingface/transformers to commit dfe30827b8ebdd974eb7ce69c7d5d8cf8e6cf852
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 14.3 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.1.dev0-py3-none-any.whl size=11184345 sha256=69fc30802a33b24e215faea1ab23bbbd13621351fe0e7a4f3c6b64a0a44f3d64
  Stored in directory: /tmp/pip-ephem-wheel-cache-dh2eehx6/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 140.9 MB/s eta 0:00:00


In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = "fewshot-datasets-new"
RESULTS_DIR = "experiments-results/5way-kshot"
QUERIES_PER_CLASS = 20
NUM_EPISODES = 3
K_VALUES = [1, 2, 3]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# HTTP session for efficient requests
_HTTP_SESSION = requests.Session()
_HTTP_SESSION.headers.update({'User-Agent': 'Mozilla/5.0'})

###Prompts

In [ ]:
# System message
FEWSHOT_5WAY_SYSTEM_MESSAGE = """You are an expert at visual geolocation. Your task is to classify indoor room images by country based on visual cues like electrical outlets, architectural styles, furniture, and regional design elements."""

# Preamble template
FEWSHOT_5WAY_PREAMBLE_TEMPLATE = """I will show you example room images from 5 different countries.
Each country has {k} example image(s). Study the visual patterns that distinguish these countries.
Then classify a query image into one of these 5 countries.

THE 5 COUNTRIES ARE: {country_list}

EXAMPLE IMAGES:
"""

# Query introduction
FEWSHOT_5WAY_QUERY_INTRO = "\n" + "="*50 + "\nQUERY IMAGE TO CLASSIFY:\n"

# Classification prompt - asks only for country
FEWSHOT_5WAY_CLASSIFICATION_PROMPT = """Based on the example images you've seen, classify this query image into one of the 5 countries.

Look for visual similarities with the examples: electrical outlets, furniture styles, architectural features, decor, and any regional indicators.
If any example helped identify visual patterns, note which one.

IMPORTANT: You MUST choose exactly one of these 5 countries: {country_list}

Provide your answer in this XML format:

<classification>
  <country>COUNTRY NAME (must be one of: {country_list})</country>
  <confidence>high/medium/low</confidence>
  <example_used>none OR example number (e.g., "Example 2") that most helped</example_used>
  <reasoning>Brief explanation of visual clues that led to this classification</reasoning>
</classification>
"""

###Data Loading

In [ ]:
def load_hotel50k_metadata(train_path: str = "hotel50k/hotel50k_geocoded_train.json",
                           test_path: str = "hotel50k/hotel50k_geocoded_test.json") -> List[Dict]:
    """Load and merge Hotel50K train and test metadata files."""
    metadata = []

    # Load train
    if Path(train_path).exists():
        print(f"Loading Hotel50K train from: {train_path}")
        with open(train_path, 'r') as f:
            train_data = json.load(f)
        print(f"  Train: {len(train_data):,} entries")
        metadata.extend(train_data)
    else:
        print(f"  Warning: Train file not found: {train_path}")

    # Load test
    if Path(test_path).exists():
        print(f"Loading Hotel50K test from: {test_path}")
        with open(test_path, 'r') as f:
            test_data = json.load(f)
        print(f"  Test: {len(test_data):,} entries")
        metadata.extend(test_data)
    else:
        print(f"  Warning: Test file not found: {test_path}")

    print(f"Total: {len(metadata):,} entries")
    return metadata


def load_image_index(index_path: str = "hotel50k/image_path_index_full_cleaned.json") -> Dict[str, str]:
    """
    Load the image index mapping image_id to local file paths.

    The index maps image IDs to absolute paths like:
    "2591644": "/content/drive/MyDrive/NLP/Thesis/images/train/subset_00/2591644.jpg"
    """
    print(f"Loading image index from: {index_path}")

    if index_path.endswith('.pkl'):
        with open(index_path, 'rb') as f:
            index = pickle.load(f)
    else:
        with open(index_path, 'r') as f:
            index = json.load(f)

    print(f"Loaded {len(index):,} image mappings")
    return index


def analyze_country_distribution(metadata: List[Dict]) -> Tuple[Counter, List[Tuple[str, int]]]:
    """Analyze the distribution of countries in the Hotel50K dataset."""
    country_counts = Counter()

    for entry in metadata:
        location = entry.get('location_derived', {})
        country = location.get('country_name')

        if country and country.lower() not in ['unknown', 'none', '']:
            country_counts[country] += 1

    sorted_countries = country_counts.most_common()

    print("Country distribution analysis")
    for i, (country, count) in enumerate(sorted_countries[:20], 1):
        marker = " <-- Exp 1" if i <= 5 else (" <-- Exp 2" if i <= 10 else (" <-- Exp 3" if i <= 15 else ""))
        print(f"{i:2}. {country}: {count:,} images{marker}")
    print("="*50)

    return country_counts, sorted_countries


def get_experiment_countries(sorted_countries: List[Tuple[str, int]], experiment_num: int) -> List[str]:
    """Get the 5 countries for a specific experiment."""
    start_idx = (experiment_num - 1) * 5
    end_idx = start_idx + 5

    countries = [country for country, count in sorted_countries[start_idx:end_idx]]

    print(f"\nExperiment {experiment_num} Countries (ranks {start_idx+1}-{end_idx}):")
    for i, country in enumerate(countries, start_idx + 1):
        count = sorted_countries[i-1][1]
        print(f"  {i}. {country}: {count:,} images")

    return countries

def validate_image(image_id: str, image_index: Dict[str, str], validate_can_open: bool = False) -> bool:
    """
    Validate that an image exists and can be loaded.e
    """
    local_path = image_index.get(str(image_id))
    if not local_path:
        return False

    if not Path(local_path).exists():
        return False

    if validate_can_open:
        try:
            with Image.open(local_path) as img:
                img.verify()
            return True
        except Exception:
            return False

    return True


def filter_samples_by_countries(metadata: List[Dict],
                                 countries: List[str],
                                 image_index: Dict[str, str]) -> Dict[str, List[Dict]]:
    """
    Filter and organize samples by country.
    Only include valid samples.
    """
    samples_by_country = {country: [] for country in countries}
    skipped_no_coords = 0
    skipped_no_url = 0
    skipped_not_in_index = 0

    for entry in tqdm(metadata, desc="Filtering samples"):
        location = entry.get('location_derived', {})
        country = location.get('country_name')

        if country not in countries:
            continue

        # Validate coordinates exist
        lat = entry.get('latitude')
        lon = entry.get('longitude')
        if lat is None or lon is None:
            skipped_no_coords += 1
            continue

        try:
            lat_val = float(lat)
            lon_val = float(lon)
            if not (-90 <= lat_val <= 90 and -180 <= lon_val <= 180):
                skipped_no_coords += 1
                continue
        except (TypeError, ValueError):
            skipped_no_coords += 1
            continue

        image_url = entry.get('image_url')
        if not image_url:
            skipped_no_url += 1
            continue

        image_id = str(entry.get('image_id'))

        # Check if image exists in local index (fast dictionary lookup)
        local_path = image_index.get(image_id)
        if not local_path:
            skipped_not_in_index += 1
            continue

        sample = {
            'image_id': int(image_id),
            'image_url': image_url,
            'local_path': local_path,
            'source': 'hotel50k',
            'metadata': {
                'image_id': int(image_id),
                'image_url': image_url,
                'latitude': lat_val,
                'longitude': lon_val,
                'continent': location.get('continent'),
                'country': country,
                'state_admin1': location.get('state_admin1'),
                'city_name': location.get('city_name'),
            }
        }

        samples_by_country[country].append(sample)

    print(f"\nFiltered samples by country:")
    for country in countries:
        print(f"  {country}: {len(samples_by_country[country]):,} samples")
    print(f"\nSkipped:")
    print(f"  Invalid/missing coordinates: {skipped_no_coords:,}")
    print(f"  Missing URL: {skipped_no_url:,}")
    print(f"  Not in local index: {skipped_not_in_index:,}")

    return samples_by_country

###Dataset Creation Funcs

In [ ]:
def check_and_fix_query_set_duplicates(query_samples: List[Dict],
                                        query_ids_by_country: Dict[str, List[int]],
                                        samples_by_country: Dict[str, List[Dict]],
                                        random_state: int = 42) -> Tuple[List[Dict], Dict[str, List[int]], int]:
    """
    Check for duplicate image IDs in query set and replace them if found.
    """
    rng = random.Random(random_state + 999)  # Different seed for replacement selection

    # Find all duplicate image IDs
    all_ids = [s['image_id'] for s in query_samples]
    id_counts = Counter(all_ids)
    duplicate_ids = {img_id for img_id, count in id_counts.items() if count > 1}

    if not duplicate_ids:
        print("  No duplicates found in query set.")
        return query_samples, query_ids_by_country, 0

    print(f"  Found {len(duplicate_ids)} duplicate image IDs in query set. Fixing...")

    # Track all used IDs (to avoid creating new duplicates)
    used_ids = set(all_ids)

    # Create a mutable copy
    fixed_query_samples = query_samples.copy()
    fixed_query_ids_by_country = {country: ids.copy() for country, ids in query_ids_by_country.items()}

    num_fixed = 0

    # For each duplicate, keep the first occurrence and replace subsequent ones
    seen_ids = set()
    for idx, sample in enumerate(fixed_query_samples):
        img_id = sample['image_id']

        if img_id in seen_ids:
            # Duplicate, needs replacement
            country = sample['ground_truth_country']

            # Find available replacements from the same country
            available = [s for s in samples_by_country[country]
                        if s['image_id'] not in used_ids]

            if not available:
                print(f"    Warning: No replacement available for duplicate {img_id} in {country}")
                continue

            # Pick a random replacement
            rng.shuffle(available)
            replacement = available[0]

            # Update the query sample
            fixed_query_samples[idx] = {
                'image_id': replacement['image_id'],
                'image_url': replacement['image_url'],
                'local_path': replacement.get('local_path'),
                'source': replacement['source'],
                'ground_truth_country': country,
                'metadata': replacement['metadata']
            }

            # Update tracking
            old_id = img_id
            new_id = replacement['image_id']
            used_ids.add(new_id)

            # Update query_ids_by_country
            if old_id in fixed_query_ids_by_country[country]:
                idx_in_country = fixed_query_ids_by_country[country].index(old_id)
                fixed_query_ids_by_country[country][idx_in_country] = new_id

            num_fixed += 1
            print(f"    Replaced duplicate {old_id} with {new_id} for {country}")
        else:
            seen_ids.add(img_id)

    print(f"  Fixed {num_fixed} duplicate(s).")
    return fixed_query_samples, fixed_query_ids_by_country, num_fixed


def create_query_set(samples_by_country: Dict[str, List[Dict]],
                     queries_per_class: int,
                     random_state: int = 42) -> Tuple[List[Dict], Dict[str, List[int]]]:
    """Create a fixed query set with equal representation from each country."""
    rng = random.Random(random_state)

    query_samples = []
    query_ids_by_country = {}

    for country, samples in samples_by_country.items():
        available = samples.copy()
        rng.shuffle(available)

        n_queries = min(queries_per_class, len(available))
        selected = available[:n_queries]

        query_ids_by_country[country] = [s['image_id'] for s in selected]

        for sample in selected:
            query_entry = {
                'image_id': sample['image_id'],
                'image_url': sample['image_url'],
                'local_path': sample.get('local_path'),
                'source': sample['source'],
                'ground_truth_country': country,
                'metadata': sample['metadata']
            }
            query_samples.append(query_entry)

    # Check and fix any duplicates
    query_samples, query_ids_by_country, num_fixed = check_and_fix_query_set_duplicates(
        query_samples, query_ids_by_country, samples_by_country, random_state
    )

    rng.shuffle(query_samples)

    print(f"\nCreated query set with {len(query_samples)} samples:")
    for country in samples_by_country.keys():
        count = sum(1 for q in query_samples if q['ground_truth_country'] == country)
        print(f"  {country}: {count} queries")

    # Final verification
    all_ids = [s['image_id'] for s in query_samples]
    if len(all_ids) != len(set(all_ids)):
        print(f"Warning: Still have duplicates after fix attempt!")
    else:
        print(f"Verified: All {len(all_ids)} query images are unique.")

    return query_samples, query_ids_by_country


def create_support_set(samples_by_country: Dict[str, List[Dict]],
                       query_ids_by_country: Dict[str, List[int]],
                       k: int,
                       episode: int,
                       random_state: int = 42) -> List[Dict]:
    """Create a support set with K examples per country, ensuring no duplicates."""
    rng = random.Random(random_state + episode * 1000)

    support_samples = []
    countries = list(samples_by_country.keys())

    # Track all used IDs across countries to prevent cross-country duplicates
    used_ids = set()

    # Also exclude all query IDs
    all_query_ids = set()
    for country_ids in query_ids_by_country.values():
        all_query_ids.update(country_ids)

    for country in countries:
        # Exclude query images and already-used support images
        available = [s for s in samples_by_country[country]
                     if s['image_id'] not in all_query_ids and s['image_id'] not in used_ids]

        # Remove duplicates within available (keep first occurrence)
        seen_in_available = set()
        deduped_available = []
        for s in available:
            if s['image_id'] not in seen_in_available:
                seen_in_available.add(s['image_id'])
                deduped_available.append(s)
        available = deduped_available

        rng.shuffle(available)

        if len(available) < k:
            print(f"Warning: Only {len(available)} samples available for {country}, need {k}")

        selected = available[:k]

        for i, sample in enumerate(selected):
            used_ids.add(sample['image_id'])

            support_entry = {
                'image_id': sample['image_id'],
                'image_url': sample['image_url'],
                'local_path': sample.get('local_path'),
                'source': sample['source'],
                'country': country,
                'example_number': len(support_samples) + 1,
                'country_example_number': i + 1,
                'metadata': sample['metadata']
            }
            support_samples.append(support_entry)

    # Sort by country to group examples
    support_samples.sort(key=lambda x: (countries.index(x['country']), x['country_example_number']))

    for i, sample in enumerate(support_samples):
        sample['example_number'] = i + 1

    # Final verification
    all_support_ids = [s['image_id'] for s in support_samples]
    if len(all_support_ids) != len(set(all_support_ids)):
        print(f"  WARNING: Duplicates in support set!")

    return support_samples


def create_5way_kshot_dataset(samples_by_country: Dict[str, List[Dict]],
                               countries: List[str],
                               k: int,
                               episode: int,
                               query_samples: List[Dict],
                               query_ids_by_country: Dict[str, List[int]],
                               random_state: int = 42) -> List[Dict]:

    """Create a complete 5-way K-shot dataset for one episode."""
    support_samples = create_support_set(
        samples_by_country,
        query_ids_by_country,
        k=k,
        episode=episode,
        random_state=random_state
    )

    dataset = []

    for query in query_samples:
        entry = {
            'query': query,
            'support_samples': support_samples,
            'countries': countries,
            'k': k,
            'n_way': len(countries),
            'n_support': len(support_samples),
            'episode': episode,
            'ground_truth_country': query['ground_truth_country']
        }
        dataset.append(entry)

    return dataset


def create_all_datasets_for_experiment(experiment_num: int,
                                        samples_by_country: Dict[str, List[Dict]],
                                        countries: List[str],
                                        output_base_dir: str,
                                        queries_per_class: int = QUERIES_PER_CLASS,
                                        num_episodes: int = NUM_EPISODES,
                                        k_values: List[int] = K_VALUES,
                                        random_state: int = RANDOM_SEED) -> Dict:

    """Create all datasets for one experiment."""
    exp_dir = Path(output_base_dir) / f"experiment_{experiment_num}"
    exp_dir.mkdir(parents=True, exist_ok=True)

    print(f"Creating datasets for Experiment {experiment_num}")
    print(f"Countries: {', '.join(countries)}")

    query_samples, query_ids_by_country = create_query_set(
        samples_by_country,
        queries_per_class=queries_per_class,
        random_state=random_state + experiment_num * 100
    )

    # Save query set
    query_set_path = exp_dir / "query_set.json"
    with open(query_set_path, 'w') as f:
        json.dump({
            'experiment': experiment_num,
            'countries': countries,
            'queries_per_class': queries_per_class,
            'total_queries': len(query_samples),
            'queries': query_samples
        }, f, indent=2)
    print(f"\nSaved query set to: {query_set_path}")

    all_datasets = {}

    for k in k_values:
        for episode in range(1, num_episodes + 1):
            print(f"\nCreating {k}-shot dataset, episode {episode}...")

            dataset = create_5way_kshot_dataset(
                samples_by_country=samples_by_country,
                countries=countries,
                k=k,
                episode=episode,
                query_samples=query_samples,
                query_ids_by_country=query_ids_by_country,
                random_state=random_state
            )

            dataset_path = exp_dir / f"5way_{k}shot_episode{episode}.json"

            dataset_output = {
                'metadata': {
                    'experiment': experiment_num,
                    'countries': countries,
                    'n_way': 5,
                    'k_shot': k,
                    'episode': episode,
                    'n_support': k * 5,
                    'n_queries': len(query_samples),
                    'random_seed': random_state
                },
                'samples': dataset
            }

            with open(dataset_path, 'w') as f:
                json.dump(dataset_output, f, indent=2)

            print(f"  Saved: {dataset_path}")
            all_datasets[(k, episode)] = dataset_path

    return {
        'experiment': experiment_num,
        'countries': countries,
        'query_set': query_samples,
        'datasets': all_datasets
    }


def create_all_5way_kshot_datasets():
    """Main function to create all datasets for all experiments."""
    print("5-Way K-shot Dataset Creation")

    metadata = load_hotel50k_metadata()
    image_index = load_image_index()

    country_counts, sorted_countries = analyze_country_distribution(metadata)

    all_experiments = {}

    for exp_num in [1, 2, 3]:
        countries = get_experiment_countries(sorted_countries, exp_num)

        samples_by_country = filter_samples_by_countries(
            metadata, countries, image_index
        )

        exp_result = create_all_datasets_for_experiment(
            experiment_num=exp_num,
            samples_by_country=samples_by_country,
            countries=countries,
            output_base_dir=OUTPUT_DIR
        )

        all_experiments[exp_num] = exp_result

    # Save global summary
    global_summary_path = Path(OUTPUT_DIR) / "all_experiments_summary.json"
    global_summary = {
        'description': '5-Way K-Shot Few-Shot Classification for VLM Geolocation',
        'task': 'country_classification',
        'n_way': 5,
        'k_values': K_VALUES,
        'num_episodes': NUM_EPISODES,
        'queries_per_class': QUERIES_PER_CLASS,
        'experiments': {
            exp_num: {'countries': result['countries']}
            for exp_num, result in all_experiments.items()
        }
    }

    with open(global_summary_path, 'w') as f:
        json.dump(global_summary, f, indent=2)

    print("Dataset creation complete.")
    print(f"Output directory: {OUTPUT_DIR}")

    return all_experiments

###Base Class

In [ ]:
class BaseGeolocationTester(ABC):
    """Abstract base class for VLM testers for 5-way country classification."""

    def __init__(self,
                 model_name: str,
                 device: str = "cuda",
                 mode: str = "auto",
                 hotel50k_image_index: Optional[Union[str, Dict[str, str]]] = None):

        self.model_name = model_name
        self.device = device
        self.mode = mode

        self.hotel50k_index = self._load_image_index(hotel50k_image_index, "Hotel50k")

        self.pipe = None
        self.model = None
        self.processor = None
        self.tokenizer = None

        print(f"Initializing {self.__class__.__name__} with {model_name}")
        self._initialize_model()

    @abstractmethod
    def _initialize_model(self):
        pass

    @abstractmethod
    def _format_messages_5way(self, query_image_url: str,
                               support_samples: List[Dict],
                               countries: List[str],
                               k: int) -> Any:
        pass

    @abstractmethod
    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        pass

    def _load_image_index(self, index_source, dataset_name: str) -> Dict[str, str]:
        if index_source is None:
            return {}
        if isinstance(index_source, dict):
            return index_source
        index_path = Path(index_source)
        if not index_path.exists():
            return {}
        try:
            if index_path.suffix == '.json':
                with open(index_path, 'r') as f:
                    return json.load(f)
            elif index_path.suffix == '.pkl':
                with open(index_path, 'rb') as f:
                    return pickle.load(f)
        except Exception:
            return {}
        return {}

    def _resize_image(self, image: Image.Image, max_size: int = 336) -> Image.Image:
        if max(image.size) > max_size:
            ratio = max_size / max(image.size)
            new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
            return image.resize(new_size, Image.LANCZOS)
        return image

    def _load_image_from_url(self, url: str, image_id: Optional[int] = None) -> Image.Image:
        """
        Load image from URL or local path.

        Priority:
        1. Try URL download first (fastest)
        2. If URL fails, fall back to local index
        """
        errors = []

        # Try URL first (fastest)
        if url.startswith(('http://', 'https://')):
            try:
                response = _HTTP_SESSION.get(url, timeout=15)
                response.raise_for_status()
                return Image.open(BytesIO(response.content)).convert('RGB')
            except Exception as e:
                errors.append(f"URL failed: {str(e)[:50]}")

        # Fallback: Check local index
        if image_id is not None:
            local_path = self.hotel50k_index.get(str(image_id))
            if local_path:
                if Path(local_path).exists():
                    try:
                        return Image.open(local_path).convert('RGB')
                    except Exception as e:
                        errors.append(f"Local file error: {e}")
                else:
                    errors.append(f"Local path not found: {local_path}")
            else:
                errors.append(f"No index entry for id {image_id}")

        # Fallback: Try as local path
        if not url.startswith(('http://', 'https://')):
            if Path(url).exists():
                try:
                    return Image.open(url).convert('RGB')
                except Exception as e:
                    errors.append(f"Direct path error: {e}")

        raise Exception(f"Could not load image: {url} (id: {image_id}). Errors: {'; '.join(errors)}")

    def _format_support_label(self, sample: Dict) -> str:
        """Format support sample label - just the country name."""
        return f"Country: {sample['country']}"

    def _build_preamble(self, countries: List[str], k: int) -> str:
        country_list = ", ".join(countries)
        return FEWSHOT_5WAY_PREAMBLE_TEMPLATE.format(k=k, country_list=country_list)

    def _build_classification_prompt(self, countries: List[str]) -> str:
        country_list = ", ".join(countries)
        return FEWSHOT_5WAY_CLASSIFICATION_PROMPT.format(country_list=country_list)

    def generate_5way_prediction(self,
                                  query_image_url: str,
                                  query_image_id: int,
                                  support_samples: List[Dict],
                                  countries: List[str],
                                  k: int,
                                  max_new_tokens: int = 512) -> str:
        formatted_input = self._format_messages_5way(
            query_image_url, support_samples, countries, k
        )
        response = self._run_inference(formatted_input, max_new_tokens)
        return response

    def parse_classification_response(self, response: str, valid_countries: List[str]) -> Dict:
        """Parse classification XML response."""
        if not response:
            return self._empty_prediction()

        # Try to find XML classification block
        patterns = [
            r'<classification>(.*?)</classification>',
            r'```xml\s*<classification>(.*?)</classification>\s*```',
        ]

        xml_content = None
        for pattern in patterns:
            match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
            if match:
                xml_content = match.group(1)
                break

        result = self._empty_prediction()

        if xml_content:
            # Clean XML
            xml_content = re.sub(r'&(?!(amp|lt|gt|quot|apos);)', r'&amp;', xml_content)

            # Extract fields
            result['country'] = self._extract_field(xml_content, 'country')
            result['confidence'] = self._extract_field(xml_content, 'confidence')
            result['example_used'] = self._extract_field(xml_content, 'example_used')
            result['reasoning'] = self._extract_field(xml_content, 'reasoning')

            # Normalize country to one of the valid options
            if result['country']:
                result['country_normalized'] = self._normalize_country(result['country'], valid_countries)
                result['country_valid'] = result['country_normalized'] is not None

            result['xml_parsed'] = True

        # Fallback: try to find any valid country in response
        if not result.get('country_normalized'):
            for country in valid_countries:
                if country.lower() in response.lower():
                    result['country_normalized'] = country
                    result['country_valid'] = True
                    break

        return result

    def _empty_prediction(self) -> Dict:
        return {
            'country': None,
            'country_normalized': None,
            'country_valid': False,
            'confidence': None,
            'example_used': None,
            'reasoning': None,
            'xml_parsed': False
        }

    def _extract_field(self, xml_content: str, field_name: str) -> Optional[str]:
        match = re.search(f'<{field_name}>(.*?)</{field_name}>', xml_content, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
        return None

    def _normalize_country(self, predicted: str, valid_countries: List[str]) -> Optional[str]:
        predicted_lower = predicted.lower().strip()

        # Direct match
        for country in valid_countries:
            if country.lower() == predicted_lower:
                return country

        # Common aliases
        aliases = {
            'usa': 'United States', 'u.s.': 'United States',
            'u.s.a.': 'United States', 'united states of america': 'United States',
            'america': 'United States',
            'uk': 'United Kingdom', 'britain': 'United Kingdom',
            'great britain': 'United Kingdom', 'england': 'United Kingdom',
        }

        if predicted_lower in aliases:
            alias_country = aliases[predicted_lower]
            if alias_country in valid_countries:
                return alias_country

        # Substring match
        for country in valid_countries:
            if country.lower() in predicted_lower or predicted_lower in country.lower():
                return country

        # Fuzzy match
        best_match = None
        best_ratio = 0.0
        for country in valid_countries:
            ratio = SequenceMatcher(None, predicted_lower, country.lower()).ratio()
            if ratio > best_ratio and ratio > 0.7:
                best_ratio = ratio
                best_match = country

        return best_match

###Model Implementations

In [ ]:
class GemmaGeolocationTester(BaseGeolocationTester):
    def _initialize_model(self):
        print("  Loading Gemma 3 4B...")
        self.pipe = pipeline(
            "image-text-to-text",
            model=self.model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

    def _format_messages_5way(self, query_image_url, support_samples, countries, k):

        user_content = [{"type": "text", "text": self._build_preamble(countries, k)}]

        # Add support examples with pre-loaded images
        for i, support in enumerate(support_samples, 1):
            img = self._load_image_from_url(support['image_url'], support.get('image_id'))
            img = self._resize_image(img, max_size=512)

            user_content.append({"type": "image", "image": img})
            location_text = f"\nExample {i}: {self._format_support_label(support)}\n"
            user_content.append({"type": "text", "text": location_text})

        query_img = self._load_image_from_url(query_image_url)
        query_img = self._resize_image(query_img, max_size=512)

        user_content.extend([
            {"type": "text", "text": FEWSHOT_5WAY_QUERY_INTRO},
            {"type": "image", "image": query_img},
            {"type": "text", "text": f"\n{self._build_classification_prompt(countries)}"}
        ])

        return [
            {"role": "system", "content": [{"type": "text", "text": FEWSHOT_5WAY_SYSTEM_MESSAGE}]},
            {"role": "user", "content": user_content}
        ]

    def _run_inference(self, formatted_input, max_new_tokens):
        output = self.pipe(text=formatted_input, max_new_tokens=max_new_tokens)
        return output[0]["generated_text"][-1]["content"].strip()


class QwenGeolocationTester(BaseGeolocationTester):

    def _initialize_model(self):
        print("  Loading Qwen 2.5-VL 3B...")
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.model_name, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
        ).to(self.device)
        self.model.eval()

        self.processor = AutoProcessor.from_pretrained(
            self.model_name, min_pixels=256*28*28, max_pixels=512*28*28
        )
        print("Qwen ready")

    def _format_messages_5way(self, query_image_url, support_samples, countries, k):

        content = [{"type": "text", "text": self._build_preamble(countries, k)}]

        for i, support in enumerate(support_samples, 1):
            img = self._load_image_from_url(support['image_url'], support.get('image_id'))
            img = self._resize_image(img, max_size=512)

            content.append({"type": "image", "image": img})
            location_text = f"\nExample {i}: {self._format_support_label(support)}\n"
            content.append({"type": "text", "text": location_text})

         query_img = self._load_image_from_url(query_image_url)
        query_img = self._resize_image(query_img, max_size=512)

        content.extend([
            {"type": "text", "text": FEWSHOT_5WAY_QUERY_INTRO},
            {"type": "image", "image": query_img},
            {"type": "text", "text": f"\n{self._build_classification_prompt(countries)}"}
        ])

        return [{"role": "user", "content": content}]

    def _run_inference(self, formatted_input, max_new_tokens):
        text = self.processor.apply_chat_template(formatted_input, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(formatted_input)

        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(self.device)

        with torch.inference_mode():
            output_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        response = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        del inputs, output_ids
        torch.cuda.empty_cache()

        return response.strip()

class MiniCPMVGeolocationTester(BaseGeolocationTester):

    def _initialize_model(self):
        print("Loading MiniCPM-V-2.6...")

        model_name_to_load = self.model_name.replace('-int4', '')

        self.model = AutoModel.from_pretrained(
            model_name_to_load,
            trust_remote_code=True,
            attn_implementation='sdpa',
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True
        )

        self.model = self.model.to(self.device)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name_to_load,
            trust_remote_code=True
        )

        # Verify model loaded properly
        sample_param = next(self.model.parameters())
        if sample_param.is_meta:
            raise RuntimeError("MiniCPM model failed to load - still on meta device")

        print(f"MiniCPM ready on {sample_param.device}")

        if torch.cuda.is_available():
            vram_used = torch.cuda.memory_allocated() / 1e9
            vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"  VRAM: {vram_used:.1f}GB / {vram_total:.1f}GB")

    def _format_messages_5way(self, query_image_url, support_samples, countries, k):
        """Format for MiniCPM, uses its specific multi-image format."""
        return {
            'query_image_url': query_image_url,
            'support_samples': support_samples,
            'countries': countries,
            'k': k
        }

    def _run_inference(self, formatted_input, max_new_tokens):
        support_samples = formatted_input['support_samples']
        query_image_url = formatted_input['query_image_url']
        countries = formatted_input['countries']
        k = formatted_input['k']

        # "Chat with multiple images" approach (all images in one message)
        content = []

        # add preample and support images with their labels
        content.append(self._build_preamble(countries, k))
        for i, support in enumerate(support_samples, 1):
            img = self._load_image_from_url(support['image_url'], support.get('image_id'))
            img = self._resize_image(img, max_size=512)
            content.append(img)
            content.append(f"\nExample {i}: {self._format_support_label(support)}\n")

        content.append(FEWSHOT_5WAY_QUERY_INTRO)

        query_img = self._load_image_from_url(query_image_url)
        query_img = self._resize_image(query_img, max_size=512)
        content.append(query_img)

        content.append(f"\n{self._build_classification_prompt(countries)}")

        msgs = [{'role': 'user', 'content': content}]

        response = self.model.chat(
            image=None,
            msgs=msgs,
            tokenizer=self.tokenizer,
            sampling=False,
            max_new_tokens=max_new_tokens
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return response.strip()

###Evaluation Funcs

In [ ]:
def calculate_5way_statistics(results: List[Dict], countries: List[str]) -> Dict:
    """Calculate classification statistics for 5-way experiments."""
    total = len(results)

    # Primary metric: country classification accuracy
    country_correct = sum(1 for r in results if r.get('evaluation', {}).get('country_correct', False))

    # Per-class statistics
    class_stats = {country: {'correct': 0, 'total': 0} for country in countries}

    # Example usage tracking
    example_usage = {'used': 0, 'not_used': 0, 'unclear': 0}

    # Confidence distribution
    confidence_dist = {'high': 0, 'medium': 0, 'low': 0, 'unknown': 0}

    for r in results:
        gt = r.get('ground_truth_country')
        if gt in class_stats:
            class_stats[gt]['total'] += 1
            if r.get('evaluation', {}).get('country_correct', False):
                class_stats[gt]['correct'] += 1

        # Track example usage
        example_used = r.get('parsed', {}).get('example_used', '')
        if example_used:
            example_lower = example_used.lower()
            if example_lower and example_lower not in ['none', 'n/a', 'na', '']:
                example_usage['used'] += 1
            elif example_lower in ['none', 'n/a', 'na']:
                example_usage['not_used'] += 1
            else:
                example_usage['unclear'] += 1

        # Track confidence
        conf = r.get('parsed', {}).get('confidence', '')
        if conf:
            conf_lower = conf.lower()
            if conf_lower in confidence_dist:
                confidence_dist[conf_lower] += 1
            else:
                confidence_dist['unknown'] += 1

    # Calculate per-class accuracy
    class_accuracy = {}
    for country, stats in class_stats.items():
        if stats['total'] > 0:
            class_accuracy[country] = stats['correct'] / stats['total']
        else:
            class_accuracy[country] = 0.0

    # XML parsing success
    xml_parsed = sum(1 for r in results if r.get('parsed', {}).get('xml_parsed', False))

    # Errors
    errors = sum(1 for r in results if 'error' in r)

    stats = {
        # Primary metric
        'country_accuracy': country_correct / total if total > 0 else 0.0,
        'country_correct': country_correct,
        'total_samples': total,

        # Per-class
        'class_accuracy': class_accuracy,
        'class_stats': class_stats,

        # Example usage
        'example_usage': {
            'examples_cited': example_usage['used'],
            'no_example_cited': example_usage['not_used'],
            'unclear': example_usage['unclear'],
            'citation_rate': example_usage['used'] / (sum(example_usage.values()) or 1)
        },

        # Confidence
        'confidence_distribution': confidence_dist,

        # Meta
        'xml_parse_success_rate': xml_parsed / total if total > 0 else 0.0,
        'errors': errors
    }

    return stats

K-Shot Test Func

In [ ]:
def run_5way_kshot_test(tester: BaseGeolocationTester,
                        dataset_path: str,
                        output_dir: str = RESULTS_DIR,
                        max_new_tokens: int = 512) -> Dict:
    """Run 5-way K-shot country classification test."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    print(f"\nLoading dataset: {dataset_path}")
    with open(dataset_path, 'r') as f:
        dataset_data = json.load(f)

    metadata = dataset_data['metadata']
    samples = dataset_data['samples']
    countries = metadata['countries']
    k = metadata['k_shot']
    episode = metadata['episode']
    experiment = metadata['experiment']

    print(f"Experiment {experiment}: {', '.join(countries)}")
    print(f"5-way {k}-shot, Episode {episode}")
    print(f"Samples: {len(samples)}")

    results = []

    print(f"\nRunning 5-way {k}-shot classification...")
    print(f"Model: {tester.model_name}")

    for idx, sample in enumerate(tqdm(samples, desc=f"5-way {k}-shot")):
        if torch.cuda.is_available() and idx % 5 == 0:
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        try:
            query = sample['query']
            support_samples = sample['support_samples']
            ground_truth_country = sample['ground_truth_country']

            response = tester.generate_5way_prediction(
                query_image_url=query['image_url'],
                query_image_id=query['image_id'],
                support_samples=support_samples,
                countries=countries,
                k=k,
                max_new_tokens=max_new_tokens
            )

            parsed = tester.parse_classification_response(response, countries)

            # Evaluate
            country_correct = (parsed.get('country_normalized') is not None and
                              parsed['country_normalized'].lower() == ground_truth_country.lower())

            evaluation = {
                'country_correct': country_correct,
                'predicted_country': parsed.get('country_normalized'),
                'ground_truth_country': ground_truth_country
            }

            results.append({
                'sample_id': idx,
                'query_image_id': query['image_id'],
                'query_image_url': query['image_url'],
                'ground_truth_country': ground_truth_country,
                'raw_response': response,
                'parsed': parsed,
                'evaluation': evaluation
            })

        except Exception as e:
            print(f"\nError at sample {idx}: {e}")
            import traceback
            traceback.print_exc()

            results.append({
                'sample_id': idx,
                'query_image_id': sample['query']['image_id'],
                'ground_truth_country': sample['ground_truth_country'],
                'error': str(e)
            })

    # Calculate statistics
    stats = calculate_5way_statistics(results, countries)

    # Save final results
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    model_name_short = tester.model_name.split('/')[-1]
    final_output = output_path / f"{model_name_short}_5way_{k}shot_exp{experiment}_ep{episode}_{timestamp}.json"

    output_data = {
        'metadata': {
            'model': tester.model_name,
            'model_class': tester.__class__.__name__,
            'experiment': experiment,
            'countries': countries,
            'n_way': 5,
            'k_shot': k,
            'episode': episode,
            'n_samples': len(samples),
            'timestamp': timestamp
        },
        'statistics': stats,
        'results': results
    }

    with open(final_output, 'w') as f:
        json.dump(output_data, f, indent=2)

    print(f"\nResults saved: {final_output}")
    _print_statistics(stats, countries, k)

    return output_data

def _print_statistics(stats: Dict, countries: List[str], k: int):
    print(f"5-Way {k}-Shot Classification Results")

    print(f"\nOverall Accuracy: {stats['country_accuracy']:.1%} ({stats['country_correct']}/{stats['total_samples']})")

    print(f"\nPer-Class Accuracy:")
    for country in countries:
        acc = stats['class_accuracy'].get(country, 0.0)
        cs = stats['class_stats'].get(country, {})
        print(f"  {country}: {acc:.1%} ({cs.get('correct', 0)}/{cs.get('total', 0)})")

    if stats.get('example_usage'):
        eu = stats['example_usage']
        print(f"\nExample Usage:")
        print(f"  Examples cited: {eu['examples_cited']}")
        print(f"  No example cited: {eu['no_example_cited']}")
        print(f"  Citation rate: {eu['citation_rate']:.1%}")

    if stats.get('errors', 0) > 0:
        print(f"\Errors: {stats['errors']} samples failed")

###Model Istantiation

In [ ]:
def get_gemma_tester():
    return GemmaGeolocationTester(
        model_name="google/gemma-3-4b-it",
        mode="pipeline",
        hotel50k_image_index="hotel50k/image_path_index_full_cleaned.json"
    )


def get_qwen_tester():
    return QwenGeolocationTester(
        model_name="Qwen/Qwen2.5-VL-3B-Instruct",
        mode="model",
        hotel50k_image_index="hotel50k/image_path_index_full_cleaned.json"
    )


def get_minicpm_tester():
    return MiniCPMVGeolocationTester(
        model_name="openbmb/MiniCPM-V-2_6",
        mode="model",
        hotel50k_image_index="hotel50k/image_path_index_full_cleaned.json"
    )

###Run Experiments Func

In [ ]:
def run_all_experiments_for_model(model_name: str):

    testers = {
        "gemma": get_gemma_tester,
        "qwen": get_qwen_tester,
        "minicpm": get_minicpm_tester
    }

    if model_name not in testers:
        raise ValueError(f"Unknown model: {model_name}. Choose from: {list(testers.keys())}")

    tester = testers[model_name]()

    for experiment in [1, 2, 3]:
        for k in K_VALUES:
            for episode in range(1, NUM_EPISODES + 1):
                dataset_path = f"{OUTPUT_DIR}/experiment_{experiment}/5way_{k}shot_episode{episode}.json"

                if not Path(dataset_path).exists():
                    print(f"Skip (not found): {dataset_path}")
                    continue

                print(f"\n{'='*60}")
                print(f"Experiment {experiment}, K={k}, Episode {episode}")
                print(f"{'='*60}")

                try:
                    run_5way_kshot_test(
                        tester=tester,
                        dataset_path=dataset_path,
                        output_dir=f"{RESULTS_DIR}/experiment_{experiment}/{model_name}"
                    )
                except Exception as e:
                    print(f"Error: {e}")
                    import traceback
                    traceback.print_exc()

                torch.cuda.empty_cache()
                gc.collect()

###Usage

Step 1: Create datasets

In [ ]:
# all_experiments = create_all_5way_kshot_datasets()

5-WAY K-SHOT DATASET CREATION
Loading Hotel50K train from: hotel50k/hotel50k_geocoded_train.json
  Train: 1,124,214 entries
Loading Hotel50K test from: hotel50k/hotel50k_geocoded_test.json
  Test: 16,171 entries
Total: 1,140,385 entries
Loading image index from: hotel50k/image_path_index_full_cleaned.json
Loaded 873,401 image mappings

COUNTRY DISTRIBUTION ANALYSIS
 1. United States: 463,484 images <-- Exp 1
 2. Italy: 60,689 images <-- Exp 1
 3. Spain: 43,524 images <-- Exp 1
 4. France: 41,300 images <-- Exp 1
 5. United Kingdom: 37,302 images <-- Exp 1
 6. Thailand: 34,207 images <-- Exp 2
 7. Australia: 33,441 images <-- Exp 2
 8. Canada: 30,546 images <-- Exp 2
 9. Greece: 24,977 images <-- Exp 2
10. Japan: 23,343 images <-- Exp 2
11. Germany: 22,009 images <-- Exp 3
12. Mexico: 18,373 images <-- Exp 3
13. China: 17,150 images <-- Exp 3
14. Indonesia: 15,936 images <-- Exp 3
15. Türkiye: 15,453 images <-- Exp 3
16. India: 15,410 images
17. Brazil: 11,926 images
18. Portugal: 9,544

Filtering samples: 100%|██████████| 1140385/1140385 [00:04<00:00, 255708.23it/s]



Filtered samples by country:
  United States: 322,621 samples
  Italy: 52,472 samples
  Spain: 37,242 samples
  France: 33,090 samples
  United Kingdom: 27,948 samples

Skipped:
  Invalid/missing coordinates: 0
  Missing URL: 0
  Not in local index: 172,926

Creating datasets for Experiment 1
Countries: United States, Italy, Spain, France, United Kingdom
  Found 1 duplicate image IDs in query set. Fixing...
    Replaced duplicate 9484574 with 6000865 for United Kingdom
  Fixed 1 duplicate(s).

Created query set with 100 samples:
  United States: 20 queries
  Italy: 20 queries
  Spain: 20 queries
  France: 20 queries
  United Kingdom: 20 queries
  Verified: All 100 query images are unique.

Saved query set to: fewshot-datasets-new/experiment_1/query_set.json

Creating 1-shot dataset, episode 1...
  Saved: fewshot-datasets-new/experiment_1/5way_1shot_episode1.json

Creating 1-shot dataset, episode 2...
  Saved: fewshot-datasets-new/experiment_1/5way_1shot_episode2.json

Creating 1-shot 

Filtering samples: 100%|██████████| 1140385/1140385 [00:00<00:00, 1226183.88it/s]



Filtered samples by country:
  Thailand: 30,053 samples
  Australia: 28,895 samples
  Canada: 24,163 samples
  Greece: 22,674 samples
  Japan: 20,933 samples

Skipped:
  Invalid/missing coordinates: 0
  Missing URL: 0
  Not in local index: 19,796

Creating datasets for Experiment 2
Countries: Thailand, Australia, Canada, Greece, Japan
  No duplicates found in query set.

Created query set with 100 samples:
  Thailand: 20 queries
  Australia: 20 queries
  Canada: 20 queries
  Greece: 20 queries
  Japan: 20 queries
  Verified: All 100 query images are unique.

Saved query set to: fewshot-datasets-new/experiment_2/query_set.json

Creating 1-shot dataset, episode 1...
  Saved: fewshot-datasets-new/experiment_2/5way_1shot_episode1.json

Creating 1-shot dataset, episode 2...
  Saved: fewshot-datasets-new/experiment_2/5way_1shot_episode2.json

Creating 1-shot dataset, episode 3...
  Saved: fewshot-datasets-new/experiment_2/5way_1shot_episode3.json

Creating 2-shot dataset, episode 1...
  Sav

Filtering samples: 100%|██████████| 1140385/1140385 [00:00<00:00, 1508089.93it/s]



Filtered samples by country:
  Germany: 16,802 samples
  Mexico: 15,537 samples
  China: 14,384 samples
  Indonesia: 14,095 samples
  Türkiye: 13,455 samples

Skipped:
  Invalid/missing coordinates: 0
  Missing URL: 0
  Not in local index: 14,648

Creating datasets for Experiment 3
Countries: Germany, Mexico, China, Indonesia, Türkiye
  Found 2 duplicate image IDs in query set. Fixing...
    Replaced duplicate 9484575 with 7001984 for Mexico
    Replaced duplicate 9484575 with 2702072 for Mexico
    Replaced duplicate 9484583 with 7092680 for Türkiye
  Fixed 3 duplicate(s).

Created query set with 100 samples:
  Germany: 20 queries
  Mexico: 20 queries
  China: 20 queries
  Indonesia: 20 queries
  Türkiye: 20 queries
  Verified: All 100 query images are unique.

Saved query set to: fewshot-datasets-new/experiment_3/query_set.json

Creating 1-shot dataset, episode 1...
  Saved: fewshot-datasets-new/experiment_3/5way_1shot_episode1.json

Creating 1-shot dataset, episode 2...
  Saved: fe

Step 2: Run experiments for each model

In [ ]:
# For Gemma:
# run_all_experiments_for_model("gemma")

Initializing GemmaGeolocationTester with google/gemma-3-4b-it
  Loading Gemma 3 4B...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

5-way 2-shot:  51%|█████     | 51/100 [09:46<09:44, 11.93s/it]Traceback (most recent call last):
  File "/tmp/ipython-input-543882044.py", line 39, in run_5way_kshot_test
    response = tester.generate_5way_prediction(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4287918308.py", line 127, in generate_5way_prediction
    formatted_input = self._format_messages_5way(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4190870709.py", line 36, in _format_messages_5way
    query_img = self._load_image_from_url(query_image_url)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4287918308.py", line 106, in _load_image_from_url
    raise Exception(f"Could not load image: {url} (id: {image_id}). Errors: {'; '.join(errors)}")
Exception: Could not load image: https://i.travelapi.com/hotels/1000000/30000/28500/28468/4556f797_b.jpg (id: None). Errors: URL failed: 404 Client Error: Not Found for url: https://


Error at sample 51: Could not load image: https://i.travelapi.com/hotels/1000000/30000/28500/28468/4556f797_b.jpg (id: None). Errors: URL failed: 404 Client Error: Not Found for url: https://i.tra


5-way 2-shot: 100%|██████████| 100/100 [19:12<00:00, 11.53s/it]



Results saved: experiments-results/5way-kshot/experiment_2/gemma/gemma-3-4b-it_5way_2shot_exp2_ep1_20260128_040136.json

5-Way 2-Shot Classification Results

OVERALL ACCURACY: 73.0% (73/100)

PER-CLASS ACCURACY:
  Thailand: 50.0% (10/20)
  Australia: 75.0% (15/20)
  Canada: 65.0% (13/20)
  Greece: 80.0% (16/20)
  Japan: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 99
  No example cited: 0
  Citation rate: 100.0%

ERRORS: 1 samples failed

Experiment 2, K=2, Episode 2

Loading dataset: fewshot-datasets-new/experiment_2/5way_2shot_episode2.json
Experiment 2: Thailand, Australia, Canada, Greece, Japan
5-way 2-shot, Episode 2
Samples: 100

Running 5-way 2-shot classification...
Model: google/gemma-3-4b-it


5-way 2-shot:  38%|███▊      | 38/100 [07:27<12:36, 12.20s/it]

Qwen Done

In [ ]:
# For Qwen:
# run_all_experiments_for_model("qwen")

Initializing QwenGeolocationTester with Qwen/Qwen2.5-VL-3B-Instruct
  Loading Qwen 2.5-VL 3B...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

  Qwen ready

Experiment 1, K=1, Episode 1

Loading dataset: fewshot-datasets-new/experiment_1/5way_1shot_episode1.json
Experiment 1: United States, Italy, Spain, France, United Kingdom
5-way 1-shot, Episode 1
Samples: 100

Running 5-way 1-shot classification...
Model: Qwen/Qwen2.5-VL-3B-Instruct


5-way 1-shot:  29%|██▉       | 29/100 [02:42<06:19,  5.35s/it]Traceback (most recent call last):
  File "/tmp/ipython-input-543882044.py", line 39, in run_5way_kshot_test
    response = tester.generate_5way_prediction(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4287918308.py", line 127, in generate_5way_prediction
    formatted_input = self._format_messages_5way(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1474727824.py", line 90, in _format_messages_5way
    query_img = self._load_image_from_url(query_image_url)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4287918308.py", line 106, in _load_image_from_url
    raise Exception(f"Could not load image: {url} (id: {image_id}). Errors: {'; '.join(errors)}")
Exception: Could not load image: https://i.travelapi.com/hotels/1000000/60000/53600/53585/b542a1f0_b.jpg (id: None). Errors: URL failed: 404 Client Error: Not Found for url: https://

...


Results saved: experiments-results/5way-kshot/experiment_3/qwen/Qwen2.5-VL-3B-Instruct_5way_3shot_exp3_ep1_20260127_051636.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 40.0% (40/100)

PER-CLASS ACCURACY:
  Germany: 30.0% (6/20)
  Mexico: 20.0% (4/20)
  China: 10.0% (2/20)
  Indonesia: 40.0% (8/20)
  Türkiye: 100.0% (20/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Experiment 3, K=3, Episode 2

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode2.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 2
Samples: 100

Running 5-way 3-shot classification...
Model: Qwen/Qwen2.5-VL-3B-Instruct


5-way 3-shot: 100%|██████████| 100/100 [12:07<00:00,  7.28s/it]



Results saved: experiments-results/5way-kshot/experiment_3/qwen/Qwen2.5-VL-3B-Instruct_5way_3shot_exp3_ep2_20260127_052844.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 36.0% (36/100)

PER-CLASS ACCURACY:
  Germany: 25.0% (5/20)
  Mexico: 10.0% (2/20)
  China: 5.0% (1/20)
  Indonesia: 40.0% (8/20)
  Türkiye: 100.0% (20/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Experiment 3, K=3, Episode 3

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode3.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 3
Samples: 100

Running 5-way 3-shot classification...
Model: Qwen/Qwen2.5-VL-3B-Instruct


5-way 3-shot: 100%|██████████| 100/100 [13:19<00:00,  8.00s/it]



Results saved: experiments-results/5way-kshot/experiment_3/qwen/Qwen2.5-VL-3B-Instruct_5way_3shot_exp3_ep3_20260127_054205.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 24.0% (24/100)

PER-CLASS ACCURACY:
  Germany: 0.0% (0/20)
  Mexico: 5.0% (1/20)
  China: 0.0% (0/20)
  Indonesia: 15.0% (3/20)
  Türkiye: 100.0% (20/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%


MiniCPM Done

In [ ]:
# For MiniCPM:
# run_all_experiments_for_model("minicpm")

Initializing MiniCPMVGeolocationTester with openbmb/MiniCPM-V-2_6
  Loading MiniCPM-V-2.6...


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

configuration_minicpm.py:   0%|          | 0.00/3.28k [00:00<?, ?B/s]

modeling_navit_siglip.py:   0%|          | 0.00/41.9k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/openbmb/MiniCPM-V-2_6:
- modeling_navit_siglip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openbmb/MiniCPM-V-2_6:
- configuration_minicpm.py
- modeling_navit_siglip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


...


Results saved: experiments-results/5way-kshot/experiment_3/minicpm/MiniCPM-V-2_6_5way_3shot_exp3_ep1_20260127_190139.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 40.0% (40/100)

PER-CLASS ACCURACY:
  Germany: 100.0% (20/20)
  Mexico: 40.0% (8/20)
  China: 15.0% (3/20)
  Indonesia: 35.0% (7/20)
  Türkiye: 10.0% (2/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Experiment 3, K=3, Episode 2

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode2.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 2
Samples: 100

Running 5-way 3-shot classification...
Model: openbmb/MiniCPM-V-2_6


5-way 3-shot: 100%|██████████| 100/100 [18:45<00:00, 11.26s/it]



Results saved: experiments-results/5way-kshot/experiment_3/minicpm/MiniCPM-V-2_6_5way_3shot_exp3_ep2_20260127_192025.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 41.0% (41/100)

PER-CLASS ACCURACY:
  Germany: 95.0% (19/20)
  Mexico: 35.0% (7/20)
  China: 10.0% (2/20)
  Indonesia: 55.0% (11/20)
  Türkiye: 10.0% (2/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Experiment 3, K=3, Episode 3

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode3.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 3
Samples: 100

Running 5-way 3-shot classification...
Model: openbmb/MiniCPM-V-2_6


5-way 3-shot: 100%|██████████| 100/100 [21:48<00:00, 13.09s/it]



Results saved: experiments-results/5way-kshot/experiment_3/minicpm/MiniCPM-V-2_6_5way_3shot_exp3_ep3_20260127_194215.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 38.0% (38/100)

PER-CLASS ACCURACY:
  Germany: 100.0% (20/20)
  Mexico: 30.0% (6/20)
  China: 15.0% (3/20)
  Indonesia: 40.0% (8/20)
  Türkiye: 5.0% (1/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%


In [ ]:
# For MiniCPM:
run_all_experiments_for_model("minicpm")

In [ ]:
"""
tester = get_gemma_tester()

results = run_5way_kshot_test(
    tester=tester,
    dataset_path="fewshot-datasets-new/experiment_3/5way_3shot_episode1.json"
)
"""

Initializing GemmaGeolocationTester with google/gemma-3-4b-it
  Loading Gemma 3 4B...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...


Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp3_ep1_20260126_153634.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 60.0% (60/100)

PER-CLASS ACCURACY:
  Germany: 65.0% (13/20)
  Mexico: 45.0% (9/20)
  China: 30.0% (6/20)
  Indonesia: 65.0% (13/20)
  Türkiye: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 98
  No example cited: 0
  Citation rate: 100.0%

ERRORS: 2 samples failed

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode2.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 2
Samples: 100

Running 5-way 3-shot classification...
Model: google/gemma-3-4b-it


5-way 3-shot: 100%|██████████| 100/100 [23:18<00:00, 13.98s/it]



Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp3_ep2_20260126_155953.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 52.0% (52/100)

PER-CLASS ACCURACY:
  Germany: 55.0% (11/20)
  Mexico: 10.0% (2/20)
  China: 40.0% (8/20)
  Indonesia: 60.0% (12/20)
  Türkiye: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode2.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 2
Samples: 100

Running 5-way 3-shot classification...
Model: google/gemma-3-4b-it


5-way 3-shot: 100%|██████████| 100/100 [23:52<00:00, 14.33s/it]


Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp3_ep2_20260126_162346.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 54.0% (54/100)

PER-CLASS ACCURACY:
  Germany: 60.0% (12/20)
  Mexico: 10.0% (2/20)
  China: 45.0% (9/20)
  Indonesia: 60.0% (12/20)
  Türkiye: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%


In [ ]:
"""
tester = get_gemma_tester()

results = run_5way_kshot_test(
    tester=tester,
    dataset_path="fewshot-datasets-new/experiment_2/5way_2shot_episode3.json"
)

ep_values_temp = ['1', '2', '3']

for ep_value in ep_values_temp:
    results = run_5way_kshot_test(
      tester=tester,
      dataset_path=f"fewshot-datasets-new/experiment_2/5way_3shot_episode{ep_value}.json"
    )
"""

Initializing GemmaGeolocationTester with google/gemma-3-4b-it
  Loading Gemma 3 4B...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...


Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp2_ep2_20260128_174819.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 73.0% (73/100)

PER-CLASS ACCURACY:
  Thailand: 65.0% (13/20)
  Australia: 60.0% (12/20)
  Canada: 70.0% (14/20)
  Greece: 75.0% (15/20)
  Japan: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 99
  No example cited: 0
  Citation rate: 100.0%

ERRORS: 1 samples failed

Loading dataset: fewshot-datasets-new/experiment_2/5way_3shot_episode3.json
Experiment 2: Thailand, Australia, Canada, Greece, Japan
5-way 3-shot, Episode 3
Samples: 100

Running 5-way 3-shot classification...
Model: google/gemma-3-4b-it


5-way 3-shot:  51%|█████     | 51/100 [11:40<12:18, 15.08s/it]Traceback (most recent call last):
  File "/tmp/ipython-input-543882044.py", line 39, in run_5way_kshot_test
    response = tester.generate_5way_prediction(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4287918308.py", line 127, in generate_5way_prediction
    formatted_input = self._format_messages_5way(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-2901358128.py", line 36, in _format_messages_5way
    query_img = self._load_image_from_url(query_image_url)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4287918308.py", line 106, in _load_image_from_url
    raise Exception(f"Could not load image: {url} (id: {image_id}). Errors: {'; '.join(errors)}")
Exception: Could not load image: https://i.travelapi.com/hotels/1000000/30000/28500/28468/4556f797_b.jpg (id: None). Errors: URL failed: 404 Client Error: Not Found for url: https://


Error at sample 51: Could not load image: https://i.travelapi.com/hotels/1000000/30000/28500/28468/4556f797_b.jpg (id: None). Errors: URL failed: 404 Client Error: Not Found for url: https://i.tra


5-way 3-shot: 100%|██████████| 100/100 [22:15<00:00, 13.36s/it]


Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp2_ep3_20260128_181035.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 76.0% (76/100)

PER-CLASS ACCURACY:
  Thailand: 75.0% (15/20)
  Australia: 80.0% (16/20)
  Canada: 50.0% (10/20)
  Greece: 85.0% (17/20)
  Japan: 90.0% (18/20)

EXAMPLE USAGE:
  Examples cited: 99
  No example cited: 0
  Citation rate: 100.0%

ERRORS: 1 samples failed


In [ ]:
"""
shot_values = ['1', '2', '3']
ep_values = ['1', '2', '3']

for shot in shot_values:
    for ep in ep_values:
        results = run_5way_kshot_test(
            tester=tester,
            dataset_path=f"fewshot-datasets-new/experiment_3/5way_{shot}shot_episode{ep}.json"
        )
"""


Loading dataset: fewshot-datasets-new/experiment_3/5way_1shot_episode1.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 1-shot, Episode 1
Samples: 100

Running 5-way 1-shot classification...
Model: google/gemma-3-4b-it


5-way 1-shot: 100%|██████████| 100/100 [17:55<00:00, 10.75s/it]



Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_1shot_exp3_ep1_20260128_182832.json

5-Way 1-Shot Classification Results

OVERALL ACCURACY: 47.0% (47/100)

PER-CLASS ACCURACY:
  Germany: 60.0% (12/20)
  Mexico: 15.0% (3/20)
  China: 15.0% (3/20)
  Indonesia: 45.0% (9/20)
  Türkiye: 100.0% (20/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Loading dataset: fewshot-datasets-new/experiment_3/5way_1shot_episode2.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 1-shot, Episode 2
Samples: 100

Running 5-way 1-shot classification...
Model: google/gemma-3-4b-it


5-way 1-shot: 100%|██████████| 100/100 [17:32<00:00, 10.53s/it]



Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_1shot_exp3_ep2_20260128_184605.json

5-Way 1-Shot Classification Results

OVERALL ACCURACY: 56.0% (56/100)

PER-CLASS ACCURACY:
  Germany: 70.0% (14/20)
  Mexico: 35.0% (7/20)
  China: 20.0% (4/20)
  Indonesia: 55.0% (11/20)
  Türkiye: 100.0% (20/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Loading dataset: fewshot-datasets-new/experiment_3/5way_1shot_episode3.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 1-shot, Episode 3
Samples: 100

Running 5-way 1-shot classification...
Model: google/gemma-3-4b-it


...


Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp3_ep1_20260128_202229.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 63.0% (63/100)

PER-CLASS ACCURACY:
  Germany: 85.0% (17/20)
  Mexico: 30.0% (6/20)
  China: 45.0% (9/20)
  Indonesia: 60.0% (12/20)
  Türkiye: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode2.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 2
Samples: 100

Running 5-way 3-shot classification...
Model: google/gemma-3-4b-it


5-way 3-shot: 100%|██████████| 100/100 [21:20<00:00, 12.80s/it]



Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp3_ep2_20260128_204351.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 63.0% (63/100)

PER-CLASS ACCURACY:
  Germany: 85.0% (17/20)
  Mexico: 30.0% (6/20)
  China: 40.0% (8/20)
  Indonesia: 65.0% (13/20)
  Türkiye: 95.0% (19/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%

Loading dataset: fewshot-datasets-new/experiment_3/5way_3shot_episode3.json
Experiment 3: Germany, Mexico, China, Indonesia, Türkiye
5-way 3-shot, Episode 3
Samples: 100

Running 5-way 3-shot classification...
Model: google/gemma-3-4b-it


5-way 3-shot: 100%|██████████| 100/100 [23:34<00:00, 14.15s/it]


Results saved: experiments-results/5way-kshot/gemma-3-4b-it_5way_3shot_exp3_ep3_20260128_210726.json

5-Way 3-Shot Classification Results

OVERALL ACCURACY: 58.0% (58/100)

PER-CLASS ACCURACY:
  Germany: 90.0% (18/20)
  Mexico: 20.0% (4/20)
  China: 40.0% (8/20)
  Indonesia: 40.0% (8/20)
  Türkiye: 100.0% (20/20)

EXAMPLE USAGE:
  Examples cited: 100
  No example cited: 0
  Citation rate: 100.0%


In [ ]:
# Cleanup (Colab)
from google.colab import runtime
runtime.unassign()